# OTTO dissertation experiment

Run every section in order. The preferred route is to keep `otto_hourly.csv` in Google Drive and copy it to Colab local storage for faster execution. The raw JSONL should be aggregated only when the processed CSV does not already exist.

**Do not use smoke-test metrics in the dissertation. Do not upload raw data, credentials, `.pt`, `.joblib`, or `.npy` files to GitHub.**


## 1. Select a GPU runtime
Use a GPU runtime before executing the remaining cells. JSON aggregation is still CPU-bound.


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/OTTO_Dissertation')
for folder in ['data/raw', 'data/processed', 'artifacts', 'logs']:
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)
print(DRIVE_ROOT)


## 3. Clone the repository and install dependencies


In [ ]:
%cd /content
!rm -rf otto-demand-forecasting-dissertation
!git clone https://github.com/nikchey29/otto-demand-forecasting-dissertation.git
%cd /content/otto-demand-forecasting-dissertation
!python -m pip install --upgrade pip
!pip install -e '.[dev]'


## 4. Preferred data route: copy the processed CSV from Drive


In [ ]:
from pathlib import Path
processed_drive = Path('/content/drive/MyDrive/OTTO_Dissertation/data/processed/otto_hourly.csv')
processed_local = Path('/content/otto-demand-forecasting-dissertation/data/processed/otto_hourly.csv')
processed_local.parent.mkdir(parents=True, exist_ok=True)
print('Drive processed CSV exists:', processed_drive.exists())
if processed_drive.exists():
    !cp "$processed_drive" "$processed_local"
    !ls -lh "$processed_local"
else:
    print('Processed CSV not found. Use the raw-data section below.')


## 5. Raw-data route — run only when the processed CSV is missing


In [ ]:
from pathlib import Path
raw_drive = Path('/content/drive/MyDrive/OTTO_Dissertation/data/raw/otto-recsys-train.jsonl')
raw_local = Path('/content/otto-demand-forecasting-dissertation/data/raw/otto-recsys-train.jsonl')
raw_local.parent.mkdir(parents=True, exist_ok=True)
print('Drive raw JSONL exists:', raw_drive.exists())
# Uncomment the next two lines only when the raw JSONL exists in Drive.
# !cp "$raw_drive" "$raw_local"
# !ls -lh "$raw_local"


In [ ]:
# Run this cell only when data/processed/otto_hourly.csv does not exist.
# !otto-forecast aggregate \
#   --input data/raw/otto-recsys-train.jsonl \
#   --output data/processed/otto_hourly.csv \
#   --frequency 1h
# !cp data/processed/otto_hourly.csv \
#   '/content/drive/MyDrive/OTTO_Dissertation/data/processed/otto_hourly.csv'


## 6. Audit the real processed data


In [ ]:
from pathlib import Path
assert Path('data/processed/otto_hourly.csv').exists(), 'Processed CSV is missing.'
!mkdir -p artifacts
!otto-forecast audit-data \
  --input data/processed/otto_hourly.csv \
  --output artifacts/data_audit.json


## 7. Run tests and quality checks


In [ ]:
!pytest -q
!python scripts/check_quality.py


## 8. Run the synthetic smoke test — software validation only


In [ ]:
!otto-forecast make-smoke-data \
  --output data/processed/synthetic_hourly.csv \
  --hours 500
!otto-forecast research --config configs/smoke.yaml


## 9. Create a persistent Colab research configuration


In [ ]:
from pathlib import Path
import yaml
source = Path('configs/research.yaml')
config = yaml.safe_load(source.read_text())
config['data']['processed_path'] = 'data/processed/otto_hourly.csv'
config['output_dir'] = '/content/drive/MyDrive/OTTO_Dissertation/artifacts'
Path('configs/research_colab.yaml').write_text(yaml.safe_dump(config, sort_keys=False))
print(Path('configs/research_colab.yaml').read_text())


## 10. Run the full rolling-origin repeated-seed benchmark


In [ ]:
!otto-forecast research --config configs/research_colab.yaml \
  2>&1 | tee '/content/drive/MyDrive/OTTO_Dissertation/logs/research_run.log'


## 11. Run the ablation study


In [ ]:
!otto-forecast ablate --config configs/research_colab.yaml \
  2>&1 | tee '/content/drive/MyDrive/OTTO_Dissertation/logs/ablation_run.log'


## 12. Verify required evidence


In [ ]:
from pathlib import Path
research_dir = Path('/content/drive/MyDrive/OTTO_Dissertation/artifacts/research')
required = [
    'data_audit.json',
    'cv_model_ranking.csv',
    'research_metrics_raw.csv',
    'research_metrics_summary.csv',
    'research_predictions.csv',
    'statistical_comparisons.csv',
    'interval_metrics.csv',
    'selected_model_horizon_metrics.csv',
    'run_metadata.csv',
    'experiment_manifest.json',
    'model_comparison_repeated.png',
    'cv_fold_stability.png',
    'selected_model_horizon_error.png',
    'ablations/ablation_metrics_raw.csv',
    'ablations/ablation_metrics_summary.csv',
]
for item in required:
    path = research_dir / item
    print(('OK   ' if path.exists() else 'MISS '), item)


## 13. Preview the real results


In [ ]:
import pandas as pd
research_dir = '/content/drive/MyDrive/OTTO_Dissertation/artifacts/research'
ranking = pd.read_csv(f'{research_dir}/cv_model_ranking.csv')
summary = pd.read_csv(f'{research_dir}/research_metrics_summary.csv')
display(ranking)
display(summary)


## 14. Saving the final outputs

Download the `artifacts/research` folder from Drive, copy only the CSV/JSON/PNG evidence into the Mac clone, then commit and push from the Mac. Do not expose a GitHub token in this notebook.
